# Phase 3: Deep Learning Method — End-to-End U-Net for Super-Resolution

In this notebook we train a **U-Net** convolutional neural network to solve the same super-resolution inverse problem tackled in Phase 2 with FISTA.

The network is trained **end-to-end** in a supervised fashion:
$$\hat{\theta} = \arg\min_{\theta} \; \mathbb{E}_{x} \left[ \| f_\theta\bigl(\uparrow(K x + e)\bigr) - x \|_2^2 \right]$$

where:
- $x$ is the **high-resolution ground truth** (loaded from the `.pt` dataset)
- $K$ is the `DownScaling` forward operator
- $e$ is additive Gaussian noise
- $\uparrow(\cdot)$ is bilinear upsampling back to $256 \times 256$
- $f_\theta$ is the **U-Net** mapping the upsampled degraded input to the HR reconstruction

**Key design choices:**
- The degradation is applied **on-the-fly** during training (same `.pt` dataset as FISTA, no separate LR files needed)
- One model is trained **per scenario** — this makes the comparison with FISTA fair (each approach operates under the same conditions)
- **Cosine annealing LR scheduler** + **Early Stopping** prevent overfitting

## 3.1 Environment Setup and Drive Mount

In [ ]:
# --- CONFIGURATION ---
BASE_DRIVE_FOLDER = 'CI_Project'
# ----------------------

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_PATH = os.path.join('/content/drive/MyDrive', BASE_DRIVE_FOLDER)
else:
    DRIVE_PATH = './CI_Project'

DATASET_PATH = os.path.join(DRIVE_PATH, 'dataset')   # same as notebooks 01 & 02
WEIGHTS_PATH = os.path.join(DRIVE_PATH, 'weights')
OUTPUT_PATH  = os.path.join(DRIVE_PATH, 'outputs', 'unet')

os.makedirs(WEIGHTS_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH,  exist_ok=True)

print(f"Drive path  : {DRIVE_PATH}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Weights path: {WEIGHTS_PATH}")
print(f"Output path : {OUTPUT_PATH}")

In [ ]:
import sys

# Clone or update the IPPy fork
if not os.path.exists('IPPy'):
    !git clone https://github.com/alessandrocampedelli/IPPy.git
else:
    %cd IPPy
    !git pull
    %cd ..

# Install extra dependencies
!pip install -q PyWavelets astra-toolbox

# Add IPPy to the Python path
sys.path.insert(0, os.path.abspath('IPPy'))

# Verify import
from IPPy import operators
from IPPy.nn import models
from IPPy.utilities import metrics
from IPPy.utilities._utilities import gaussian_noise
print("IPPy library imported successfully!")

## 3.2 Hyperparameters

Modify `SCALE_FACTOR` and `NOISE_LEVEL` to select the degradation scenario for training.

In [ ]:
import torch

# ─── Scenario (one model per scenario) ──────────────────────────────────────
SCALE_FACTOR = 2        # Downscaling factor  (2 or 4)
NOISE_LEVEL  = 0.01     # Gaussian noise σ   (0.005 or 0.01)

# ─── Dataset ────────────────────────────────────────────────────────────────
IMG_SHAPE    = (256, 256)

# ─── U-Net architecture ─────────────────────────────────────────────────────
MIDDLE_CH  = [32, 64, 128, 256]   # Channel progression — 4 encoder levels
N_LAYERS   = 2                    # Conv layers per block
FINAL_ACT  = 'sigmoid'            # Ensures output in [0, 1]

# ─── Training ───────────────────────────────────────────────────────────────
N_EPOCHS   = 20
BATCH_SIZE = 8
LR         = 1e-3
SAVE_EACH  = 5          # Save checkpoint every N epochs

# ─── Early Stopping ─────────────────────────────────────────────────────────
ES_PATIENCE = 8         # Stop if val loss does not improve for ES_PATIENCE epochs
ES_DELTA    = 1e-5      # Minimum improvement to be considered significant

# ─── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Derived
SCENARIO_TAG = f's{SCALE_FACTOR}_sig{str(NOISE_LEVEL).replace(".", "")}'
WEIGHTS_FILE     = os.path.join(WEIGHTS_PATH, f'unet_{SCENARIO_TAG}_weights.pth')
CHECKPOINT_FILE  = os.path.join(WEIGHTS_PATH, f'unet_{SCENARIO_TAG}_ckpt.pth')

print(f"Device       : {DEVICE}")
print(f"Scenario     : s={SCALE_FACTOR}, σ={NOISE_LEVEL}  (tag: {SCENARIO_TAG})")
print(f"Epochs       : {N_EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}")
print(f"Early stop   : patience={ES_PATIENCE}")
print(f"Weights file     : {WEIGHTS_FILE}")
print(f"Checkpoint file  : {CHECKPOINT_FILE}")

## 3.3 On-the-Fly Dataset

We define a `DegradedDataset` that:
1. Loads a ground-truth HR tensor `x` from the `.pt` file (shape `(1, 256, 256)`)
2. Applies the forward model on-the-fly: `y = K(x) + noise`
3. Bilinearly upsamples `y` back to `256×256`
4. Returns the pair `(y_up, x)` — input and target for the U-Net

This is identical to the data pipeline used in Phase 2 with FISTA, ensuring a fair comparison.

In [ ]:
import glob as glob_module
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPPy.operators import DownScaling


class DegradedDataset(Dataset):
    """
    Loads HR tensors from .pt files and applies the forward model on-the-fly.

    Returns:
        (y_up, x_true) where:
          y_up   : degraded image bilinearly upsampled to IMG_SHAPE  — network input
          x_true : original HR image                                  — network target
    """
    def __init__(self, data_dir: str, scale_factor: int, noise_level: float,
                 img_shape=(256, 256), augment: bool = False):
        self.files       = sorted(glob_module.glob(os.path.join(data_dir, '*.pt')))
        assert len(self.files) > 0, f"No .pt files found in {data_dir}"
        self.K           = DownScaling(img_shape=img_shape,
                                       downscale_factor=scale_factor, mode='avg')
        self.noise_level = noise_level
        self.img_shape   = img_shape
        self.augment     = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # x_true : (1, 256, 256)
        x_true = torch.load(self.files[idx])

        # Optional horizontal flip for data augmentation (training only)
        if self.augment and torch.rand(1).item() > 0.5:
            x_true = torch.flip(x_true, dims=[-1])

        # Forward model: y = K(x) + noise  →  shape (1, H/s, W/s)
        x_batch  = x_true.unsqueeze(0)                          # (1, 1, 256, 256)
        y_clean  = self.K(x_batch)                              # (1, 1, H/s, W/s)
        y_noisy  = y_clean + gaussian_noise(y_clean, self.noise_level)

        # Bilinear upsample to original resolution
        y_up = F.interpolate(y_noisy, size=self.img_shape, mode='bilinear',
                             align_corners=False)               # (1, 1, 256, 256)
        y_up = y_up.squeeze(0).clamp(0.0, 1.0)                 # (1, 256, 256)

        return y_up, x_true


# ── Build dataloaders ────────────────────────────────────────────────────────
train_dataset = DegradedDataset(os.path.join(DATASET_PATH, 'train'),
                                SCALE_FACTOR, NOISE_LEVEL, augment=True)
val_dataset   = DegradedDataset(os.path.join(DATASET_PATH, 'val'),
                                SCALE_FACTOR, NOISE_LEVEL, augment=False)
test_dataset  = DegradedDataset(os.path.join(DATASET_PATH, 'test'),
                                SCALE_FACTOR, NOISE_LEVEL, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f"Train samples : {len(train_dataset)}")
print(f"Val   samples : {len(val_dataset)}")
print(f"Test  samples : {len(test_dataset)}")

# Quick sanity check
y_s, x_s = train_dataset[0]
print(f"\nInput  shape : {y_s.shape}  range [{y_s.min():.3f}, {y_s.max():.3f}]")
print(f"Target shape : {x_s.shape}  range [{x_s.min():.3f}, {x_s.max():.3f}]")

## 3.4 Model Instantiation

In [ ]:
import torch.nn as nn
from IPPy.nn.models import UNet

model = UNet(
    ch_in              = 1,
    ch_out             = 1,
    middle_ch          = MIDDLE_CH,
    n_layers_per_block = N_LAYERS,
    down_layers        = ('ResDownBlock',) * (len(MIDDLE_CH) - 1),
    up_layers          = ('ResUpBlock',)   * (len(MIDDLE_CH) - 1),
    final_activation   = FINAL_ACT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net ready on {DEVICE}")
print(f"  Trainable parameters : {n_params:,}")
print(f"  Channel progression  : {MIDDLE_CH}")
print(f"  Encoder/decoder depth: {len(MIDDLE_CH)-1} levels")
print(f"  Final activation     : {FINAL_ACT}")

## 3.5 Training with Early Stopping

- **Optimiser**: Adam with initial LR = `LR`
- **Scheduler**: Cosine Annealing — smoothly decreases LR to `eta_min=1e-5` over `T_max` epochs
- **Early Stopping**: halts training if the validation loss does not improve by more than `ES_DELTA` for `ES_PATIENCE` consecutive epochs; the best weights are restored automatically
- **Auto-load**: if `WEIGHTS_FILE` already exists on Drive, training is skipped entirely

In [ ]:
import time
import copy
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# ── Determine training status ──────────────────────────────────────────────
ckpt = torch.load(CHECKPOINT_FILE, map_location=DEVICE, weights_only=False) if os.path.exists(CHECKPOINT_FILE) else None

if ckpt is not None and ckpt.get('completed', False):
    # All epochs already done — load final weights and skip
    model.load_state_dict(torch.load(WEIGHTS_FILE, map_location=DEVICE, weights_only=True))
    model.eval()
    history = ckpt['history']
    SKIP_TRAINING = True
    print(f"✅ Training already complete ({ckpt['epoch']}/{N_EPOCHS} epochs).")
    print(f"   Best val_loss = {ckpt['best_val_loss']:.5f}")
    print(f"   Weights loaded from {WEIGHTS_FILE}")
elif ckpt is not None and not ckpt.get('completed', False):
    # Interrupted — resume from checkpoint
    SKIP_TRAINING = False
    START_EPOCH   = ckpt['epoch'] + 1
    print(f"🔄 Resuming from epoch {START_EPOCH}/{N_EPOCHS}  "
          f"(best val_loss so far: {ckpt['best_val_loss']:.5f})")
else:
    # No checkpoint — fresh start
    SKIP_TRAINING = False
    START_EPOCH   = 1
    print(f"🆕 Training from scratch for {N_EPOCHS} epochs.")

if not SKIP_TRAINING:
    loss_fn   = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=N_EPOCHS, eta_min=1e-5
    )

    # ── Restore state from checkpoint if resuming ────────────────────────────
    if ckpt is not None:
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        history       = ckpt['history']
        best_val_loss = ckpt['best_val_loss']
        best_weights  = ckpt['best_model_state_dict']
        es_counter    = ckpt['es_counter']
    else:
        history       = {'train_loss': [], 'val_loss': [], 'val_psnr': [], 'val_ssim': []}
        best_val_loss = float('inf')
        best_weights  = copy.deepcopy(model.state_dict())
        es_counter    = 0

    stopped_epoch = N_EPOCHS
    print()

    for epoch in range(START_EPOCH, N_EPOCHS + 1):
        # ── Training ──────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        t0 = time.time()
        for y_batch, x_batch in tqdm(train_loader, desc=f"Epoch {epoch}/{N_EPOCHS} [Train]", leave=False):
            y_batch = y_batch.to(DEVICE)
            x_batch = x_batch.to(DEVICE)

            optimizer.zero_grad()
            pred = model(y_batch)
            loss = loss_fn(pred, x_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        scheduler.step()

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        val_loss  = 0.0
        val_psnr  = 0.0
        val_ssim  = 0.0
        n_samples = 0
        with torch.no_grad():
            for y_batch, x_batch in tqdm(val_loader, desc=f"Epoch {epoch}/{N_EPOCHS} [Val]", leave=False):
                y_batch = y_batch.to(DEVICE)
                x_batch = x_batch.to(DEVICE)
                pred = model(y_batch)
                val_loss += loss_fn(pred, x_batch).item()
                for i in range(pred.shape[0]):
                    p_i = pred[i:i+1].cpu()
                    x_i = x_batch[i:i+1].cpu()
                    val_psnr += metrics.PSNR(p_i, x_i)
                    val_ssim += metrics.SSIM(p_i, x_i)
                n_samples += pred.shape[0]

        val_loss /= len(val_loader)
        val_psnr /= n_samples
        val_ssim /= n_samples

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_psnr'].append(val_psnr)
        history['val_ssim'].append(val_ssim)

        elapsed = time.time() - t0
        print(f"Epoch [{epoch:2d}/{N_EPOCHS}]  "
              f"train={train_loss:.5f}  val={val_loss:.5f}  "
              f"PSNR={val_psnr:.2f} dB  SSIM={val_ssim:.4f}  "
              f"lr={scheduler.get_last_lr()[0]:.2e}  ({elapsed:.1f}s)")

        # ── Early stopping check ───────────────────────────────────────────────
        if val_loss < best_val_loss - ES_DELTA:
            best_val_loss = val_loss
            best_weights  = copy.deepcopy(model.state_dict())
            es_counter    = 0
        else:
            es_counter += 1
            if es_counter >= ES_PATIENCE:
                stopped_epoch = epoch
                print(f"\n🛑 Early stopping triggered at epoch {epoch} "
                      f"(no improvement for {ES_PATIENCE} epochs).")
                break

        # ── Save checkpoint after every epoch ─────────────────────────────────
        is_done = (epoch == N_EPOCHS) or (es_counter >= ES_PATIENCE)
        torch.save({
            'epoch':                 epoch,
            'completed':             is_done,
            'model_state_dict':      model.state_dict(),
            'best_model_state_dict': best_weights,
            'optimizer_state_dict':  optimizer.state_dict(),
            'scheduler_state_dict':  scheduler.state_dict(),
            'best_val_loss':         best_val_loss,
            'es_counter':            es_counter,
            'history':               history,
        }, CHECKPOINT_FILE)

    # ── Restore and save final best model ─────────────────────────────────────
    model.load_state_dict(best_weights)
    model.eval()
    torch.save(best_weights, WEIGHTS_FILE)
    best_epoch = stopped_epoch - es_counter if es_counter >= ES_PATIENCE else stopped_epoch
    print(f"\n✅ Training complete. Best val_loss={best_val_loss:.5f} "
          f"(epoch {best_epoch})")
    print(f"   Final weights saved  → {WEIGHTS_FILE}")
    print(f"   Checkpoint saved     → {CHECKPOINT_FILE}")

## 3.6 Training Curves

In [ ]:
if not SKIP_TRAINING:
    n_trained = len(history['train_loss'])
    epochs_range = range(1, n_trained + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Loss
    axes[0].plot(epochs_range, history['train_loss'], label='Train', color='royalblue')
    axes[0].plot(epochs_range, history['val_loss'],   label='Val',   color='tomato')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
    axes[0].set_title('Loss vs. Epoch')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # PSNR
    axes[1].plot(epochs_range, history['val_psnr'], color='seagreen', linewidth=1.8)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PSNR (dB)')
    axes[1].set_title('Val PSNR vs. Epoch')
    axes[1].grid(True, alpha=0.3)

    # SSIM
    axes[2].plot(epochs_range, history['val_ssim'], color='darkorange', linewidth=1.8)
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('SSIM')
    axes[2].set_title('Val SSIM vs. Epoch')
    axes[2].grid(True, alpha=0.3)

    plt.suptitle(f'U-Net Training History  —  s={SCALE_FACTOR}, σ={NOISE_LEVEL}', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, f'unet_training_{SCENARIO_TAG}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Training curves not available (pre-trained weights loaded).")

## 3.7 Quantitative Evaluation on Test Set

We evaluate the model trained on scenario `(SCALE_FACTOR, NOISE_LEVEL)` on all four degradation scenarios.
This shows how well the network generalises when the operating conditions change slightly.

In [ ]:
ALL_SCENARIOS = [
    (2, 0.005),
    (2, 0.01),
    (4, 0.005),
    (4, 0.01),
]

model.eval()
unet_results = {}   # unet_results[(s, sigma)] = {'psnr': [], 'ssim': [], 'x_hat': []}

with torch.no_grad():
    for s, sigma in ALL_SCENARIOS:
        key = (s, sigma)
        unet_results[key] = {'psnr': [], 'ssim': [], 'x_hat': []}

        scen_dataset = DegradedDataset(
            os.path.join(DATASET_PATH, 'test'), s, sigma, augment=False
        )

        print(f"\n{'='*60}")
        print(f"  Scale factor s={s}  |  Noise σ={sigma}")
        print(f"{'='*60}")

        for idx in range(len(scen_dataset)):
            y_up, x_true = scen_dataset[idx]          # (1,256,256), (1,256,256)
            y_up_b  = y_up.unsqueeze(0).to(DEVICE)    # (1, 1, 256, 256)
            x_true_b = x_true.unsqueeze(0)            # (1, 1, 256, 256) CPU

            x_hat = model(y_up_b).cpu().clamp(0.0, 1.0)  # (1, 1, 256, 256)

            psnr_val = metrics.PSNR(x_hat, x_true_b)
            ssim_val = metrics.SSIM(x_hat, x_true_b)

            unet_results[key]['psnr'].append(psnr_val)
            unet_results[key]['ssim'].append(ssim_val)
            unet_results[key]['x_hat'].append(x_hat.detach())

            print(f"  Image {idx+1:3d}/{len(scen_dataset)}  "
                  f"| PSNR={psnr_val:6.2f} dB  | SSIM={ssim_val:.4f}")

        mean_psnr = sum(unet_results[key]['psnr']) / len(unet_results[key]['psnr'])
        mean_ssim = sum(unet_results[key]['ssim']) / len(unet_results[key]['ssim'])
        unet_results[key]['mean_psnr'] = mean_psnr
        unet_results[key]['mean_ssim'] = mean_ssim
        print(f"\n  ➜  Mean PSNR = {mean_psnr:.2f} dB  |  Mean SSIM = {mean_ssim:.4f}")

## 3.8 Quantitative Results Summary

In [ ]:
print(f"\n{'─'*58}")
print(f"  {'Config':<24} {'Mean PSNR (dB)':>14} {'Mean SSIM':>12}")
print(f"{'─'*58}")
for (s, sigma), res in unet_results.items():
    label = f"s={s}, σ={sigma}"
    trained = " ← trained" if (s == SCALE_FACTOR and sigma == NOISE_LEVEL) else ""
    print(f"  {label:<24} {res['mean_psnr']:>14.2f} {res['mean_ssim']:>12.4f}{trained}")
print(f"{'─'*58}")

## 3.9 Qualitative Visualisation

For each scenario: degraded input (bilinear upsampled), U-Net reconstruction, and ground truth.

In [ ]:
IMG_IDX = 0   # Test image index to display

n_rows = len(unet_results)
fig, axes = plt.subplots(n_rows, 3, figsize=(13, 4 * n_rows))
if n_rows == 1:
    axes = [axes]

for row, ((s, sigma), res) in enumerate(unet_results.items()):
    # Rebuild degraded input for display
    scen_ds = DegradedDataset(
        os.path.join(DATASET_PATH, 'test'), s, sigma, augment=False
    )
    y_up, x_true = scen_ds[IMG_IDX]

    lr_disp  = y_up[0].numpy()                          # (256, 256) bilinear upsampled
    hr_disp  = x_true[0].numpy()                        # (256, 256) ground truth
    hat_disp = res['x_hat'][IMG_IDX][0, 0].numpy()      # (256, 256) U-Net output

    psnr_val = res['psnr'][IMG_IDX]
    ssim_val = res['ssim'][IMG_IDX]

    ax_gt, ax_deg, ax_rec = axes[row]

    ax_gt.imshow(hr_disp,  cmap='gray', vmin=0, vmax=1)
    ax_gt.set_title('Ground Truth', fontsize=11)
    ax_gt.axis('off')

    ax_deg.imshow(lr_disp, cmap='gray', vmin=0, vmax=1)
    ax_deg.set_title(f'Degraded  (s={s}, σ={sigma})', fontsize=11)
    ax_deg.axis('off')

    ax_rec.imshow(hat_disp, cmap='gray', vmin=0, vmax=1)
    ax_rec.set_title(
        f'U-Net Reconstruction\nPSNR={psnr_val:.2f} dB  SSIM={ssim_val:.4f}',
        fontsize=11
    )
    ax_rec.axis('off')

plt.suptitle(
    f'U-Net End-to-End — Qualitative Results  (trained on s={SCALE_FACTOR}, σ={NOISE_LEVEL})',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, f'unet_qualitative_{SCENARIO_TAG}.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved to {OUTPUT_PATH}")

## 3.10 Save Results to Drive

We persist metrics as a CSV and reconstructed tensors as `.pt` files.
Notebook 05 will load these alongside the FISTA results for the final comparison.

In [ ]:
import csv

# ── Save metrics CSV ─────────────────────────────────────────────────────────
csv_path = os.path.join(OUTPUT_PATH, f'unet_metrics_{SCENARIO_TAG}.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['scale_factor', 'noise_level', 'image_idx', 'psnr_db', 'ssim'])
    for (s, sigma), res in unet_results.items():
        for i, (p, ss) in enumerate(zip(res['psnr'], res['ssim'])):
            writer.writerow([s, sigma, i, round(p, 4), round(ss, 6)])

print(f"Metrics CSV saved → {csv_path}")

# ── Save reconstructed tensors ───────────────────────────────────────────────
for (s, sigma), res in unet_results.items():
    config_tag  = f's{s}_sig{str(sigma).replace(".", "")}'
    tensors_dir = os.path.join(OUTPUT_PATH, config_tag)
    os.makedirs(tensors_dir, exist_ok=True)
    for i, x_hat in enumerate(res['x_hat']):
        torch.save(x_hat, os.path.join(tensors_dir, f'unet_recon_{i:03d}.pt'))

print(f"Reconstructed tensors saved → {OUTPUT_PATH}")
print(f"\n✅ Phase 3 complete. U-Net results are ready for comparison in Notebook 05.")